# Notebook 03 — Calibrated Scenario-Based Project Delay Sensitivity Engine (Reviewer-Response Version)

This notebook treats the action engine strictly as **model-based sensitivity analysis**. It adds:

- use of the internally calibrated XGBoost probability pipeline from Notebook 02 when available;
- a publication-ready table for all 11 actions, exact variable changes, constraints, feasibility assumptions, and evidence basis;
- alternative action-magnitude and budget/duration-cap sensitivity tests;
- continued evaluation on five illustrative cases and 500 high-risk synthetic projects.

The actions are researcher-defined hypothetical scenarios. They are not causal treatment effects, validated recommendations, or estimates of implementation benefit.


In [ ]:
# =========================
# 0. Setup
# =========================
import warnings
warnings.filterwarnings("ignore")

import json
import platform
import itertools
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

# Repository-root resolver: allows notebooks to run from the repository root
# or from the notebooks/ directory without changing output paths.
def _find_repo_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".project-root").exists():
            return candidate
    return current

REPO_ROOT = _find_repo_root()
PROJECT_DIR = REPO_ROOT / "project_delay_outputs"
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR = PROJECT_DIR / "plots"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

LOW_RISK_THRESHOLD = 0.35
HIGH_RISK_THRESHOLD = 0.65
CLASSIFICATION_THRESHOLD = 0.50
MIN_PREDICTED_PROBABILITY_DROP = 0.01
MAX_COMBO_SIZE = 2
LARGE_EVAL_N = 500
ACTION_MAGNITUDE_SCALES = [0.50, 1.00, 1.50]
BUDGET_DURATION_CAPS = [0.05, 0.10]
RUN_ACTION_SENSITIVITY = True

print("Scenario engine seed:", SEED)
print("Large-sample high-risk evaluation size:", LARGE_EVAL_N)


In [ ]:
# =========================
# 1. Load Dataset, Model, and Calibration Artifacts
# =========================
data_path = DATA_DIR / "synthetic_project_delay_dataset.csv"
model_path = MODEL_DIR / "best_delay_prediction_pipeline.pkl"
metadata_path = MODEL_DIR / "model_metadata.pkl"

calibrated_base_model_path = MODEL_DIR / "scenario_probability_base_pipeline.pkl"
probability_calibrator_path = MODEL_DIR / "scenario_probability_platt_calibrator.pkl"

if not data_path.exists():
    raise FileNotFoundError("Run Notebook 01 first.")
if not model_path.exists():
    raise FileNotFoundError("Run Notebook 02 first.")
if not metadata_path.exists():
    raise FileNotFoundError("Model metadata was not found. Run Notebook 02 first.")

df = pd.read_csv(data_path)
metadata = joblib.load(metadata_path)
features = metadata["features"]

# The uncalibrated baseline model is retained for traceability.
baseline_model = joblib.load(model_path)

# Prefer the clean model-fit + calibration artifacts from Notebook 02.
if calibrated_base_model_path.exists() and probability_calibrator_path.exists():
    model = joblib.load(calibrated_base_model_path)
    probability_calibrator = joblib.load(probability_calibrator_path)
    USE_CALIBRATED_PROBABILITIES = True
    scenario_probability_source = "Platt-calibrated XGBoost from Notebook 02"
else:
    model = baseline_model
    probability_calibrator = None
    USE_CALIBRATED_PROBABILITIES = False
    scenario_probability_source = (
        "Uncalibrated baseline model (calibration artifacts not found)"
    )

print("Loaded dataset:", df.shape)
print("Baseline selected model:", metadata.get("best_model_name", "Unknown"))
print("Scenario probability source:", scenario_probability_source)
print("Number of model features:", len(features))


In [ ]:
# =========================
# 2. Prediction Helpers
# =========================
risk_order = ["Low", "Medium", "High"]

def probability_to_risk_level(prob):
    # Descriptive synthetic risk bands only; not operationally calibrated thresholds.
    if prob < LOW_RISK_THRESHOLD:
        return "Low"
    if prob < HIGH_RISK_THRESHOLD:
        return "Medium"
    return "High"

def _safe_logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

def get_delay_probabilities(project_df):
    raw = model.predict_proba(
        project_df[features].copy()
    )[:, 1]

    if USE_CALIBRATED_PROBABILITIES:
        return probability_calibrator.predict_proba(
            _safe_logit(raw)
        )[:, 1]

    return raw

def predict_project_risk(project_df):
    proba = get_delay_probabilities(project_df)

    result = project_df.copy()
    result["Predicted_Delay_Probability"] = proba
    result["Predicted_Delay_Label"] = np.where(
        proba >= CLASSIFICATION_THRESHOLD,
        "Delayed",
        "Not Delayed"
    )
    result["Predicted_Risk_Level"] = [
        probability_to_risk_level(p) for p in proba
    ]
    return result

def get_delay_probability(project_series):
    project_df = pd.DataFrame([project_series])
    return float(get_delay_probabilities(project_df)[0])


In [ ]:
# =========================
# 3. Logical Scenario Constraints
# =========================
score_columns = [
    "Scope_Clarity_Score",
    "Stakeholder_Communication_Score",
    "Coordination_Score",
    "Contractor_Experience_Score",
    "Resource_Availability_Score",
    "Labor_Productivity_Score",
    "Equipment_Availability_Score",
    "Supplier_Reliability_Score",
    "Contractor_Financial_Stability",
    "Owner_Decision_Speed_Score",
    "Weather_Risk_Score",
    "Permit_Risk_Score",
    "Inflation_Risk_Score",
    "Site_Condition_Risk_Score",
    "External_Risk_Score",
]

immutable_features = {
    "Project_ID",
    "Project_Type",
    "Sector",
    "Region",
    "Contract_Type",
    "Procurement_Method",
    "Complexity_Level",
    "Complexity_Score",
    "Force_Majeure_Flag",
    "Weather_Risk_Score",
    "Permit_Risk_Score",
    "Inflation_Risk_Score",
    "Site_Condition_Risk_Score",
    "External_Risk_Score",
}

def _clip_preserving_original(value, original_value, lower, upper):
    """Bound scenario values without silently altering an out-of-range baseline value."""
    effective_lower = min(lower, original_value)
    effective_upper = max(upper, original_value)
    return np.clip(value, effective_lower, effective_upper)

def enforce_candidate_constraints(project, original, max_increase_cap=0.10):
    """
    Apply logical constraints to an adjusted candidate only.
    The baseline project is never modified.
    """
    p = project.copy()

    for col in score_columns:
        if col in p.index and col in original.index:
            p[col] = float(_clip_preserving_original(
                float(p[col]), float(original[col]), 1.0, 10.0
            ))

    for col in ["BIM_Adoption_Level", "AI_Tools_Adoption_Level"]:
        if col in p.index and col in original.index:
            bounded = _clip_preserving_original(
                float(p[col]), float(original[col]), 0.0, 3.0
            )
            p[col] = int(round(bounded))

    p["Planned_Budget_Million"] = float(np.clip(
        float(p["Planned_Budget_Million"]),
        float(original["Planned_Budget_Million"]),
        float(original["Planned_Budget_Million"]) * (1.0 + max_increase_cap)
    ))

    p["Planned_Duration_Days"] = int(round(np.clip(
        float(p["Planned_Duration_Days"]),
        float(original["Planned_Duration_Days"]),
        float(original["Planned_Duration_Days"]) * (1.0 + max_increase_cap)
    )))

    original_auth = int(round(original["Authorities_Involved"]))
    original_days = int(round(original["Approval_Per_Authority_Days"]))

    p["Authorities_Involved"] = int(round(_clip_preserving_original(
        float(p["Authorities_Involved"]), float(original_auth), 1.0, 5.0
    )))
    p["Approval_Per_Authority_Days"] = int(round(_clip_preserving_original(
        float(p["Approval_Per_Authority_Days"]), float(original_days), 1.0, 15.0
    )))

    approval_changed = (
        p["Authorities_Involved"] != original_auth
        or p["Approval_Per_Authority_Days"] != original_days
    )

    if approval_changed:
        p["Approval_Duration_Days"] = int(
            p["Authorities_Involved"] * p["Approval_Per_Authority_Days"]
        )
    else:
        p["Approval_Duration_Days"] = original["Approval_Duration_Days"]

    for col in ["Design_Change_Count", "Change_Request_Count", "Safety_Incident_Count"]:
        if col in p.index:
            p[col] = int(max(0, round(float(p[col]))))

    if "Quality_Defect_Rate" in p.index:
        upper = max(20.0, float(original["Quality_Defect_Rate"]))
        p["Quality_Defect_Rate"] = float(
            np.clip(float(p["Quality_Defect_Rate"]), 0.0, upper)
        )

    if "Procurement_Lead_Time_Days" in p.index:
        low = min(5, int(round(original["Procurement_Lead_Time_Days"])))
        high = max(120, int(round(original["Procurement_Lead_Time_Days"])))
        p["Procurement_Lead_Time_Days"] = int(np.clip(
            round(float(p["Procurement_Lead_Time_Days"])), low, high
        ))

    if "Payment_Delay_Days" in p.index:
        high = max(90, int(round(original["Payment_Delay_Days"])))
        p["Payment_Delay_Days"] = int(np.clip(
            round(float(p["Payment_Delay_Days"])), 0, high
        ))

    if "Schedule_Buffer_Percent" in p.index:
        high = max(25.0, float(original["Schedule_Buffer_Percent"]))
        p["Schedule_Buffer_Percent"] = float(np.clip(
            float(p["Schedule_Buffer_Percent"]), 0.0, high
        ))

    for col in immutable_features:
        if col in original.index and col in p.index:
            p[col] = original[col]

    return p


In [ ]:
# =========================
# 4. Scenario Action Library and Publication Audit Table
# =========================
def _scaled_count_reduction(base_reduction, scale):
    return max(1, int(np.ceil(base_reduction * scale)))

def action_library(original, magnitude_scale=1.0, max_increase_cap=0.10):
    """
    Predefined hypothetical project-management scenarios.

    `magnitude_scale` changes the numerical magnitude of editable actions for
    sensitivity testing. These magnitudes are researcher-defined simulation
    choices, not estimated causal effects.
    """
    s = float(magnitude_scale)
    cap = float(max_increase_cap)

    return [
        {
            "Action": "Increase schedule buffer by adding controlled contingency",
            "Category": "Schedule Planning",
            "Variables_Changed": "Schedule_Buffer_Percent; Planned_Duration_Days",
            "Baseline_Change": "+5 percentage points buffer; +5% duration",
            "Constraint": f"Duration increase capped at {cap:.0%}; schedule buffer bounded",
            "Feasibility_Assumption": "Hypothetical planning action; project-specific feasibility/cost not modeled",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by project-management logic",
            "Update": lambda p: p.assign(
                Schedule_Buffer_Percent=np.minimum(
                    p["Schedule_Buffer_Percent"] + 5 * s, 25
                ),
                Planned_Duration_Days=np.minimum(
                    p["Planned_Duration_Days"] * (1 + 0.05 * s),
                    original["Planned_Duration_Days"] * (1 + cap)
                )
            )
        },
        {
            "Action": "Increase budget contingency for critical resources",
            "Category": "Budget / Resources",
            "Variables_Changed": "Planned_Budget_Million; Resource_Availability_Score; Equipment_Availability_Score",
            "Baseline_Change": "+5% budget; +1.0 resource score; +0.8 equipment score",
            "Constraint": f"Budget increase capped at {cap:.0%}; scores bounded 1–10",
            "Feasibility_Assumption": "Hypothetical funding/resource action; affordability not modeled",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by project-management logic",
            "Update": lambda p: p.assign(
                Planned_Budget_Million=np.minimum(
                    p["Planned_Budget_Million"] * (1 + 0.05 * s),
                    original["Planned_Budget_Million"] * (1 + cap)
                ),
                Resource_Availability_Score=np.minimum(
                    p["Resource_Availability_Score"] + 1.0 * s, 10
                ),
                Equipment_Availability_Score=np.minimum(
                    p["Equipment_Availability_Score"] + 0.8 * s, 10
                )
            )
        },
        {
            "Action": "Adopt BIM coordination and clash detection",
            "Category": "BIM / Digital Coordination",
            "Variables_Changed": "BIM_Adoption_Level; Coordination_Score; Design_Change_Count; Quality_Defect_Rate",
            "Baseline_Change": "+1 BIM level; +1.3 coordination; -1 design change; -1.5 defect-rate points",
            "Constraint": "BIM level 0–3; scores bounded; counts non-negative",
            "Feasibility_Assumption": "Hypothetical digital-coordination scenario; implementation readiness/cost not modeled",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                BIM_Adoption_Level=np.minimum(
                    p["BIM_Adoption_Level"] + _scaled_count_reduction(1, s), 3
                ),
                Coordination_Score=np.minimum(
                    p["Coordination_Score"] + 1.3 * s, 10
                ),
                Design_Change_Count=np.maximum(
                    p["Design_Change_Count"] - _scaled_count_reduction(1, s), 0
                ),
                Quality_Defect_Rate=np.maximum(
                    p["Quality_Defect_Rate"] - 1.5 * s, 0
                )
            )
        },
        {
            "Action": "Use AI-based early warning dashboard",
            "Category": "AI / Decision Support",
            "Variables_Changed": "AI_Tools_Adoption_Level; Owner_Decision_Speed_Score; Stakeholder_Communication_Score",
            "Baseline_Change": "+1 AI level; +0.9 decision speed; +0.7 communication",
            "Constraint": "AI level 0–3; scores bounded 1–10",
            "Feasibility_Assumption": "Hypothetical digital-support scenario; adoption effort not modeled",
            "Evidence_Basis": "Researcher-defined simulation scenario",
            "Update": lambda p: p.assign(
                AI_Tools_Adoption_Level=np.minimum(
                    p["AI_Tools_Adoption_Level"] + _scaled_count_reduction(1, s), 3
                ),
                Owner_Decision_Speed_Score=np.minimum(
                    p["Owner_Decision_Speed_Score"] + 0.9 * s, 10
                ),
                Stakeholder_Communication_Score=np.minimum(
                    p["Stakeholder_Communication_Score"] + 0.7 * s, 10
                )
            )
        },
        {
            "Action": "Improve stakeholder communication plan",
            "Category": "Communication",
            "Variables_Changed": "Stakeholder_Communication_Score; Coordination_Score",
            "Baseline_Change": "+1.5 communication; +1.0 coordination",
            "Constraint": "Scores bounded 1–10",
            "Feasibility_Assumption": "Hypothetical management-process improvement",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                Stakeholder_Communication_Score=np.minimum(
                    p["Stakeholder_Communication_Score"] + 1.5 * s, 10
                ),
                Coordination_Score=np.minimum(
                    p["Coordination_Score"] + 1.0 * s, 10
                )
            )
        },
        {
            "Action": "Reduce approval cycle through dedicated authority liaison",
            "Category": "Approvals / Governance",
            "Variables_Changed": "Authorities_Involved; Approval_Per_Authority_Days; Owner_Decision_Speed_Score; derived Approval_Duration_Days",
            "Baseline_Change": "cap authorities at 5; -2 approval days/authority; +1.0 decision-speed score",
            "Constraint": "Approval days >=1; decision score <=10; approval duration recomputed",
            "Feasibility_Assumption": "Hypothetical governance-process improvement",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                Authorities_Involved=np.minimum(p["Authorities_Involved"], 5),
                Approval_Per_Authority_Days=np.maximum(
                    p["Approval_Per_Authority_Days"]
                    - _scaled_count_reduction(2, s), 1
                ),
                Owner_Decision_Speed_Score=np.minimum(
                    p["Owner_Decision_Speed_Score"] + 1.0 * s, 10
                )
            )
        },
        {
            "Action": "Freeze scope baseline and strengthen change-control board",
            "Category": "Scope / Change Control",
            "Variables_Changed": "Scope_Clarity_Score; Change_Request_Count; Design_Change_Count",
            "Baseline_Change": "+1.5 scope clarity; -1 change request; -1 design change",
            "Constraint": "Score <=10; counts non-negative",
            "Feasibility_Assumption": "Hypothetical change-control improvement",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                Scope_Clarity_Score=np.minimum(
                    p["Scope_Clarity_Score"] + 1.5 * s, 10
                ),
                Change_Request_Count=np.maximum(
                    p["Change_Request_Count"] - _scaled_count_reduction(1, s), 0
                ),
                Design_Change_Count=np.maximum(
                    p["Design_Change_Count"] - _scaled_count_reduction(1, s), 0
                )
            )
        },
        {
            "Action": "Improve procurement plan and supplier follow-up",
            "Category": "Procurement",
            "Variables_Changed": "Supplier_Reliability_Score; Procurement_Lead_Time_Days",
            "Baseline_Change": "+1.2 supplier reliability; -7 procurement days",
            "Constraint": "Score <=10; procurement lead time >=5 days",
            "Feasibility_Assumption": "Hypothetical procurement-process improvement",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                Supplier_Reliability_Score=np.minimum(
                    p["Supplier_Reliability_Score"] + 1.2 * s, 10
                ),
                Procurement_Lead_Time_Days=np.maximum(
                    p["Procurement_Lead_Time_Days"]
                    - _scaled_count_reduction(7, s), 5
                )
            )
        },
        {
            "Action": "Strengthen contractor staffing and productivity monitoring",
            "Category": "Contractor Performance",
            "Variables_Changed": "Labor_Productivity_Score; Contractor_Experience_Score; Resource_Availability_Score",
            "Baseline_Change": "+1.2 productivity; +0.6 contractor experience; +0.8 resources",
            "Constraint": "Scores bounded 1–10",
            "Feasibility_Assumption": "Hypothetical contractor-management improvement",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                Labor_Productivity_Score=np.minimum(
                    p["Labor_Productivity_Score"] + 1.2 * s, 10
                ),
                Contractor_Experience_Score=np.minimum(
                    p["Contractor_Experience_Score"] + 0.6 * s, 10
                ),
                Resource_Availability_Score=np.minimum(
                    p["Resource_Availability_Score"] + 0.8 * s, 10
                )
            )
        },
        {
            "Action": "Reduce payment bottlenecks through faster invoice review",
            "Category": "Finance",
            "Variables_Changed": "Payment_Delay_Days; Contractor_Financial_Stability",
            "Baseline_Change": "-7 payment-delay days; +0.8 financial-stability score",
            "Constraint": "Payment delay non-negative; score <=10",
            "Feasibility_Assumption": "Hypothetical finance-process improvement",
            "Evidence_Basis": "Researcher-defined simulation scenario informed by literature-supported direction",
            "Update": lambda p: p.assign(
                Payment_Delay_Days=np.maximum(
                    p["Payment_Delay_Days"]
                    - _scaled_count_reduction(7, s), 0
                ),
                Contractor_Financial_Stability=np.minimum(
                    p["Contractor_Financial_Stability"] + 0.8 * s, 10
                )
            )
        },
        {
            "Action": "Add external-risk contingency planning",
            "Category": "External Risk Mitigation",
            "Variables_Changed": "Schedule_Buffer_Percent; Planned_Duration_Days",
            "Baseline_Change": "+4 percentage points buffer; +4% duration",
            "Constraint": f"Duration increase capped at {cap:.0%}; buffer bounded",
            "Feasibility_Assumption": "Hypothetical contingency-planning action; cost/contract impacts not modeled",
            "Evidence_Basis": "Researcher-defined simulation scenario",
            "Update": lambda p: p.assign(
                Schedule_Buffer_Percent=np.minimum(
                    p["Schedule_Buffer_Percent"] + 4 * s, 25
                ),
                Planned_Duration_Days=np.minimum(
                    p["Planned_Duration_Days"] * (1 + 0.04 * s),
                    original["Planned_Duration_Days"] * (1 + cap)
                )
            )
        },
    ]

# Publication-ready action specification at the BASELINE magnitude and 10% cap.
_action_catalogue = action_library(
    df.iloc[0],
    magnitude_scale=1.0,
    max_increase_cap=0.10
)

action_catalogue_df = pd.DataFrame([
    {
        "Action": a["Action"],
        "Category": a["Category"],
        "Variables_Changed": a["Variables_Changed"],
        "Baseline_Numerical_Change": a["Baseline_Change"],
        "Constraints": a["Constraint"],
        "Feasibility_Assumption": a["Feasibility_Assumption"],
        "Evidence_Basis": a["Evidence_Basis"],
    }
    for a in _action_catalogue
])

action_catalogue_df.to_csv(
    RESULTS_DIR / "FINAL_scenario_action_specification.csv",
    index=False
)

display(action_catalogue_df)


In [ ]:
# =========================
# 5. Scenario-Based Decision-Support Engine
# =========================
def _values_equal(a, b):
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b

def recommend_for_project(
    project_row,
    top_n=3,
    max_combo_size=MAX_COMBO_SIZE,
    min_probability_drop=MIN_PREDICTED_PROBABILITY_DROP,
    magnitude_scale=1.0,
    max_increase_cap=0.10
):
    """
    Evaluate hypothetical scenario actions for one project.
    Scenarios are ranked by model-predicted probability reduction.
    This function does not estimate causal effects.
    """
    if len(project_row) != 1:
        raise ValueError("project_row must contain exactly one project.")

    original = project_row.iloc[0].copy()

    # FIX: baseline probability is calculated from the untouched original record.
    base_prob = get_delay_probability(original)

    actions = action_library(original, magnitude_scale=magnitude_scale, max_increase_cap=max_increase_cap)
    candidate_records = []

    for combo_size in range(1, max_combo_size + 1):
        for combo in itertools.combinations(actions, combo_size):
            p = pd.DataFrame([original.copy()])
            action_names = []
            categories = []

            for action in combo:
                p = action["Update"](p)
                action_names.append(action["Action"])
                categories.append(action["Category"])

            adjusted = enforce_candidate_constraints(p.iloc[0], original, max_increase_cap=max_increase_cap)

            changed = {}
            for col in features:
                if col in adjusted.index and col in original.index:
                    if not _values_equal(adjusted[col], original[col]):
                        changed[col] = {
                            "Before": original[col],
                            "After": adjusted[col]
                        }

            candidate_records.append({
                "Adjusted_Project": adjusted,
                "Action_List": action_names,
                "Category_List": categories,
                 "Changed_Features": changed,
                "Magnitude_Scale": float(magnitude_scale),
                "Budget_Duration_Cap": float(max_increase_cap)
            })

    if not candidate_records:
        return []

    candidate_df = pd.DataFrame([
        rec["Adjusted_Project"][features]
        for rec in candidate_records
    ])

    candidate_probs = get_delay_probabilities(candidate_df)

    candidates = []

    for rec, new_prob in zip(candidate_records, candidate_probs):
        new_prob = float(new_prob)
        predicted_drop = float(base_prob - new_prob)

        if predicted_drop >= min_probability_drop:
            scenario_risk = probability_to_risk_level(new_prob)

            candidates.append({
                "Project_ID": original.get("Project_ID", "New Project"),
                "Original_Project": original.copy(),
                "Adjusted_Project": rec["Adjusted_Project"].copy(),
                "Base_Probability": float(base_prob),
                "Scenario_Probability": new_prob,
                "Recommended_Probability": new_prob,  # compatibility alias
                "Predicted_Probability_Reduction": predicted_drop,
                "Probability_Reduction": predicted_drop,  # compatibility alias
                "Base_Risk_Level": probability_to_risk_level(base_prob),
                "Scenario_Risk_Level": scenario_risk,
                "Recommended_Risk_Level": scenario_risk,  # compatibility alias
                "Scenario_Actions": " + ".join(rec["Action_List"]),
                "Actions": " + ".join(rec["Action_List"]),  # compatibility alias
                "Categories": " + ".join(rec["Category_List"]),
                "Action_List": rec["Action_List"],
                "Category_List": rec["Category_List"],
                "Changed_Features": rec["Changed_Features"],
                "Magnitude_Scale": float(magnitude_scale),
                "Budget_Duration_Cap": float(max_increase_cap)
            })

    return sorted(
        candidates,
        key=lambda x: x["Predicted_Probability_Reduction"],
        reverse=True
    )[:top_n]

def recommendations_to_table(recommendations):
    rows = []

    for i, rec in enumerate(recommendations, 1):
        changed_text = "; ".join([
            f"{k}: {v['Before']} to {v['After']}"
            for k, v in rec["Changed_Features"].items()
        ])

        rows.append({
            "Rank": i,
            "Base Probability": rec["Base_Probability"],
            "Scenario Probability": rec["Scenario_Probability"],
            "Predicted Probability Reduction": rec["Predicted_Probability_Reduction"],
            "Base Risk": rec["Base_Risk_Level"],
            "Scenario Risk": rec["Scenario_Risk_Level"],
            "Scenario Actions": rec["Scenario_Actions"],
            "Changed Features": changed_text,
            "Magnitude Scale": rec.get("Magnitude_Scale", 1.0),
            "Budget/Duration Cap": rec.get("Budget_Duration_Cap", 0.10),
            "Probability Source": scenario_probability_source
        })

    return pd.DataFrame(rows)


In [ ]:
# =========================
# 6. Illustrative High-Risk Synthetic Project
# =========================
# This is an illustrative synthetic case, not a historical real-world project.

pred_all = predict_project_risk(df)

high_risk = pred_all[
    pred_all["Predicted_Delay_Probability"] >= HIGH_RISK_THRESHOLD
].copy()

print("Number of model-predicted high-risk synthetic projects:", len(high_risk))

example_project_id = "P-408698"

if example_project_id in set(df["Project_ID"].astype(str)):
    project_row = df[df["Project_ID"].astype(str) == example_project_id].copy()
else:
    project_row = high_risk.sort_values(
        "Predicted_Delay_Probability",
        ascending=False
    ).head(1).copy()
    example_project_id = str(project_row.iloc[0]["Project_ID"])

example_prediction = predict_project_risk(project_row)

display(example_prediction[
    ["Project_ID", "Predicted_Delay_Probability",
     "Predicted_Delay_Label", "Predicted_Risk_Level"]
])

recommendations = recommend_for_project(
    project_row,
    top_n=3,
    max_combo_size=MAX_COMBO_SIZE
)

rec_table = recommendations_to_table(recommendations)
display(rec_table)

if not rec_table.empty:
    direct_prob = float(example_prediction.iloc[0]["Predicted_Delay_Probability"])
    scenario_base = float(rec_table.iloc[0]["Base Probability"])

    assert np.isclose(direct_prob, scenario_base, atol=1e-12), (
        f"Baseline mismatch: direct={direct_prob}, scenario={scenario_base}"
    )

    print("Baseline consistency check: PASSED")

rec_table.to_csv(
    RESULTS_DIR / "FINAL_illustrative_project_scenarios.csv",
    index=False
)


In [ ]:
# =========================
# 7. Five Reproducible Synthetic Example Projects
# =========================
# Retained for continuity with the original manuscript tables.

example_projects_df = df.sample(5, random_state=SEED).copy()
example_predictions = predict_project_risk(example_projects_df)

all_scenario_tables = []

for idx in example_projects_df.index:
    row = example_projects_df.loc[[idx]]

    scenarios = recommend_for_project(
        row,
        top_n=3,
        max_combo_size=MAX_COMBO_SIZE
    )

    table = recommendations_to_table(scenarios)

    if not table.empty:
        table.insert(0, "Project_ID", row.iloc[0]["Project_ID"])
        all_scenario_tables.append(table)

if all_scenario_tables:
    five_project_scenarios_df = pd.concat(
        all_scenario_tables,
        ignore_index=True
    )
else:
    five_project_scenarios_df = pd.DataFrame()

display(example_predictions[
    ["Project_ID", "Predicted_Delay_Probability",
     "Predicted_Delay_Label", "Predicted_Risk_Level"]
])

display(five_project_scenarios_df)

if not five_project_scenarios_df.empty:
    base_check = (
        five_project_scenarios_df
        .groupby("Project_ID", as_index=False)
        .first()[["Project_ID", "Base Probability"]]
        .merge(
            example_predictions[
                ["Project_ID", "Predicted_Delay_Probability"]
            ],
            on="Project_ID",
            how="left"
        )
    )

    base_check["Absolute Difference"] = (
        base_check["Base Probability"]
        - base_check["Predicted_Delay_Probability"]
    ).abs()

    display(base_check)

    assert (base_check["Absolute Difference"] < 1e-12).all(), (
        "At least one scenario baseline differs from the direct model prediction."
    )

    print("Five-project baseline consistency check: PASSED")

example_predictions[
    ["Project_ID", "Predicted_Delay_Probability",
     "Predicted_Delay_Label", "Predicted_Risk_Level"]
].to_csv(
    RESULTS_DIR / "FINAL_five_project_predictions.csv",
    index=False
)

five_project_scenarios_df.to_csv(
    RESULTS_DIR / "FINAL_five_project_scenarios.csv",
    index=False
)


In [ ]:
# =========================
# 7.1 Baseline vs Best Scenario for Five Examples
# =========================
if not five_project_scenarios_df.empty:
    best_per_project = (
        five_project_scenarios_df
        .sort_values(
            ["Project_ID", "Predicted Probability Reduction"],
            ascending=[True, False]
        )
        .groupby("Project_ID", as_index=False)
        .head(1)
    )

    plot_df = best_per_project[
        ["Project_ID", "Base Probability", "Scenario Probability"]
    ].set_index("Project_ID")

    ax = plot_df.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Baseline vs Best Scenario-Predicted Delay Probability")
    ax.set_ylabel("Model-Predicted Delay Probability")
    ax.set_ylim(0, 1)

    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / "FINAL_five_project_baseline_vs_scenario.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 8. Larger-Sample Evaluation of High-Risk Synthetic Projects

To avoid relying on only a few illustrative examples, the scenario engine is evaluated on a reproducible random sample of model-predicted high-risk synthetic projects.

The evaluation reports the proportion of projects for which at least one qualifying scenario is identified, the baseline and best-scenario probabilities, the model-predicted probability change, risk-level transitions, and the frequency of actions appearing in the best-ranked scenarios.

These results evaluate the **behavior of the scenario engine within the synthetic modeling environment**. They do not establish the real-world causal effectiveness of the actions.


In [ ]:
# =========================
# 8.1 Evaluate on a Larger High-Risk Sample
# =========================
high_risk_pool = pred_all[
    pred_all["Predicted_Delay_Probability"] >= HIGH_RISK_THRESHOLD
].copy()

eval_n = min(LARGE_EVAL_N, len(high_risk_pool))

high_risk_eval_df = high_risk_pool.sample(
    n=eval_n,
    random_state=SEED
).copy()

evaluation_rows = []

for _, row in high_risk_eval_df.iterrows():
    row_df = pd.DataFrame([row[df.columns]])

    base_prob = float(row["Predicted_Delay_Probability"])
    base_risk = probability_to_risk_level(base_prob)

    scenarios = recommend_for_project(
        row_df,
        top_n=1,
        max_combo_size=MAX_COMBO_SIZE,
        min_probability_drop=MIN_PREDICTED_PROBABILITY_DROP
    )

    if scenarios:
        best = scenarios[0]
        scenario_prob = float(best["Scenario_Probability"])
        predicted_drop = float(best["Predicted_Probability_Reduction"])
        scenario_risk = best["Scenario_Risk_Level"]
        scenario_actions = best["Scenario_Actions"]
        scenario_found = True
    else:
        scenario_prob = base_prob
        predicted_drop = 0.0
        scenario_risk = base_risk
        scenario_actions = ""
        scenario_found = False

    evaluation_rows.append({
        "Project_ID": row["Project_ID"],
        "Base_Probability": base_prob,
        "Best_Scenario_Probability": scenario_prob,
        "Predicted_Probability_Reduction": predicted_drop,
        "Base_Risk_Level": base_risk,
        "Best_Scenario_Risk_Level": scenario_risk,
        "Scenario_Found": scenario_found,
        "Best_Scenario_Actions": scenario_actions
    })

large_eval_results = pd.DataFrame(evaluation_rows)

assert (large_eval_results["Predicted_Probability_Reduction"] >= -1e-12).all()

assert np.allclose(
    large_eval_results["Predicted_Probability_Reduction"],
    large_eval_results["Base_Probability"]
    - large_eval_results["Best_Scenario_Probability"],
    atol=1e-12
)

display(large_eval_results.head(10))

large_eval_results.to_csv(
    RESULTS_DIR / "FINAL_large_sample_scenario_evaluation.csv",
    index=False
)

print("Evaluated high-risk synthetic projects:", len(large_eval_results))


In [ ]:
# =========================
# 8.2 Aggregate Large-Sample Scenario Results
# =========================
n_total = len(large_eval_results)
n_found = int(large_eval_results["Scenario_Found"].sum())

overall_summary = pd.DataFrame([{
    "Evaluated_High_Risk_Projects": n_total,
    "Projects_With_Qualifying_Scenario": n_found,
    "Scenario_Coverage_Percent": 100 * n_found / n_total if n_total else np.nan,
    "Mean_Base_Probability": large_eval_results["Base_Probability"].mean(),
    "Median_Base_Probability": large_eval_results["Base_Probability"].median(),
    "Mean_Best_Scenario_Probability": large_eval_results["Best_Scenario_Probability"].mean(),
    "Median_Best_Scenario_Probability": large_eval_results["Best_Scenario_Probability"].median(),
    "Mean_Predicted_Probability_Reduction": large_eval_results["Predicted_Probability_Reduction"].mean(),
    "Median_Predicted_Probability_Reduction": large_eval_results["Predicted_Probability_Reduction"].median(),
    "Minimum_Predicted_Probability_Reduction": large_eval_results["Predicted_Probability_Reduction"].min(),
    "Maximum_Predicted_Probability_Reduction": large_eval_results["Predicted_Probability_Reduction"].max()
}])

risk_transitions = pd.crosstab(
    large_eval_results["Base_Risk_Level"],
    large_eval_results["Best_Scenario_Risk_Level"],
    margins=True
)

action_counter = Counter()

for actions in large_eval_results.loc[
    large_eval_results["Scenario_Found"],
    "Best_Scenario_Actions"
].dropna():
    for action in str(actions).split(" + "):
        action = action.strip()
        if action:
            action_counter[action] += 1

action_frequency = pd.DataFrame(
    action_counter.most_common(),
    columns=["Scenario Action", "Count"]
)

if not action_frequency.empty:
    action_frequency["Percent_of_Evaluated_Projects"] = (
        100 * action_frequency["Count"] / n_total
    )

display(overall_summary)
display(risk_transitions)
display(action_frequency)

overall_summary.to_csv(
    RESULTS_DIR / "FINAL_large_sample_scenario_summary.csv",
    index=False
)

risk_transitions.to_csv(
    RESULTS_DIR / "FINAL_large_sample_risk_transitions.csv"
)

action_frequency.to_csv(
    RESULTS_DIR / "FINAL_large_sample_action_frequency.csv",
    index=False
)


In [ ]:
# =========================
# 8.3 Action-Magnitude and Constraint Sensitivity
# =========================
# Re-evaluate the SAME high-risk sample under alternative action magnitudes
# and budget/duration caps. Large scenario coverage in the baseline setting
# is therefore not treated as evidence of real-world effectiveness.

scenario_sensitivity_summary_df = pd.DataFrame()

if RUN_ACTION_SENSITIVITY:
    sensitivity_rows = []

    for magnitude_scale in ACTION_MAGNITUDE_SCALES:
        for cap in BUDGET_DURATION_CAPS:
            project_rows = []

            for _, row in high_risk_eval_df.iterrows():
                row_df = pd.DataFrame([row[df.columns]])
                base_prob = get_delay_probability(row_df.iloc[0])

                scenarios = recommend_for_project(
                    row_df,
                    top_n=1,
                    max_combo_size=MAX_COMBO_SIZE,
                    min_probability_drop=MIN_PREDICTED_PROBABILITY_DROP,
                    magnitude_scale=magnitude_scale,
                    max_increase_cap=cap
                )

                if scenarios:
                    best = scenarios[0]
                    scenario_prob = float(best["Scenario_Probability"])
                    reduction = float(best["Predicted_Probability_Reduction"])
                    found = True
                else:
                    scenario_prob = base_prob
                    reduction = 0.0
                    found = False

                project_rows.append({
                    "Base_Probability": base_prob,
                    "Best_Scenario_Probability": scenario_prob,
                    "Predicted_Probability_Reduction": reduction,
                    "Scenario_Found": found
                })

            tmp = pd.DataFrame(project_rows)
            sensitivity_rows.append({
                "Magnitude_Scale": magnitude_scale,
                "Budget_Duration_Cap": cap,
                "Evaluated_Projects": len(tmp),
                "Projects_With_Qualifying_Scenario": int(tmp["Scenario_Found"].sum()),
                "Scenario_Coverage_Percent": 100 * tmp["Scenario_Found"].mean(),
                "Mean_Base_Probability": tmp["Base_Probability"].mean(),
                "Mean_Best_Scenario_Probability": tmp["Best_Scenario_Probability"].mean(),
                "Mean_Predicted_Probability_Reduction": tmp["Predicted_Probability_Reduction"].mean(),
                "Median_Predicted_Probability_Reduction": tmp["Predicted_Probability_Reduction"].median(),
                "Probability_Source": scenario_probability_source,
                "Interpretation": "Synthetic model sensitivity only; not causal effectiveness"
            })

    scenario_sensitivity_summary_df = pd.DataFrame(sensitivity_rows)
    display(scenario_sensitivity_summary_df)

    scenario_sensitivity_summary_df.to_csv(
        RESULTS_DIR / "FINAL_scenario_magnitude_constraint_sensitivity.csv",
        index=False
    )
else:
    print("Action-magnitude/constraint sensitivity skipped.")


## 9. Scenario Traceability for the Illustrative Project

This section shows which feature values were changed by the best-ranked scenario and the associated change in model-predicted delay probability. It is a **traceability and sensitivity analysis**, not a causal explanation of intervention effectiveness.


In [ ]:
# =========================
# 9.1 Trace Feature Changes
# =========================
def compare_project_changes(before_series, after_series, feature_list=None):
    if feature_list is None:
        feature_list = features

    rows = []

    for col in feature_list:
        before = before_series.get(col, np.nan)
        after = after_series.get(col, np.nan)

        if pd.isna(before) and pd.isna(after):
            continue

        if not _values_equal(before, after):
            direction = "Changed"

            try:
                before_num = float(before)
                after_num = float(after)

                if after_num > before_num:
                    direction = "Increase"
                elif after_num < before_num:
                    direction = "Decrease"
            except Exception:
                pass

            rows.append({
                "Feature": col,
                "Before": before,
                "After": after,
                "Direction": direction
            })

    return pd.DataFrame(rows)

if recommendations:
    best_scenario = recommendations[0]

    before = best_scenario["Original_Project"]
    after = best_scenario["Adjusted_Project"]

    changed_features_df = compare_project_changes(before, after)

    print("Best-ranked scenario actions:")
    for action in best_scenario["Action_List"]:
        print("-", action)

    print()
    print(f"Baseline probability: {best_scenario['Base_Probability']:.4f}")
    print(f"Scenario probability: {best_scenario['Scenario_Probability']:.4f}")
    print(
        "Model-predicted probability reduction: "
        f"{best_scenario['Predicted_Probability_Reduction']:.4f}"
    )

    display(changed_features_df)

    if not changed_features_df.empty:
        plot_changes = changed_features_df.copy()

        plot_changes["Before_Num"] = pd.to_numeric(
            plot_changes["Before"], errors="coerce"
        )
        plot_changes["After_Num"] = pd.to_numeric(
            plot_changes["After"], errors="coerce"
        )

        plot_changes = plot_changes.dropna(
            subset=["Before_Num", "After_Num"]
        )

        if not plot_changes.empty:
            ax = (
                plot_changes
                .set_index("Feature")[["Before_Num", "After_Num"]]
                .plot(kind="bar", figsize=(10, 5))
            )

            ax.set_title(
                "Feature Values in Baseline and Best-Ranked Scenario"
            )
            ax.set_ylabel("Feature Value")

            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()

            plt.savefig(
                PLOTS_DIR / "FINAL_illustrative_scenario_feature_changes.png",
                dpi=300,
                bbox_inches="tight"
            )
            plt.show()

    changed_features_df.to_csv(
        RESULTS_DIR / "FINAL_illustrative_scenario_changed_features.csv",
        index=False
    )
else:
    print("No qualifying scenario was generated for the illustrative project.")


In [ ]:
# =========================
# 9.2 Rank Illustrative Scenarios
# =========================
if not rec_table.empty:
    plot_df = rec_table.head(10).copy()

    plot_df["Scenario Number"] = [
        f"Scenario {i}"
        for i in range(1, len(plot_df) + 1)
    ]

    ax = plot_df.sort_values(
        "Predicted Probability Reduction"
    ).plot(
        x="Scenario Number",
        y="Predicted Probability Reduction",
        kind="barh",
        figsize=(10, 6),
        legend=False
    )

    ax.set_title(
        "Best Scenarios Ranked by Model-Predicted Probability Reduction"
    )
    ax.set_xlabel("Model-Predicted Delay Probability Reduction")
    ax.set_ylabel("Scenario")

    plt.tight_layout()

    plt.savefig(
        PLOTS_DIR / "FINAL_illustrative_scenario_ranking.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
else:
    print("No scenario table is available to plot.")


In [ ]:
# =========================
# 10. Reproducibility Metadata
# =========================
scenario_metadata = {
    "seed": SEED,
    "dataset_records": int(len(df)),
    "model_name": metadata.get("best_model_name", "Unknown"),
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    "risk_thresholds": {
        "low_upper_bound": LOW_RISK_THRESHOLD,
        "high_lower_bound": HIGH_RISK_THRESHOLD
    },
    "minimum_predicted_probability_drop": MIN_PREDICTED_PROBABILITY_DROP,
    "maximum_action_combination_size": MAX_COMBO_SIZE,
    "large_sample_high_risk_evaluation_n": int(
        min(LARGE_EVAL_N, len(high_risk_pool))
    ),
    "number_of_action_rules": len(action_library(df.iloc[0], 1.0, 0.10)),
    "probability_source": scenario_probability_source,
    "uses_internal_probability_calibration": bool(USE_CALIBRATED_PROBABILITIES),
    "action_magnitude_scales_tested": ACTION_MAGNITUDE_SCALES,
    "budget_duration_caps_tested": BUDGET_DURATION_CAPS,
    "interpretation": (
        "Scenario-based model sensitivity analysis; "
        "outputs are not causal treatment-effect estimates."
    ),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "joblib_version": joblib.__version__
}

with open(
    RESULTS_DIR / "FINAL_scenario_engine_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(scenario_metadata, f, indent=2)

print(json.dumps(scenario_metadata, indent=2))


## 11. Applying the Engine to a New Project

A new project may be supplied as a one-row CSV containing the same model-input features used during training.

For any real-world application, the prediction model should first be validated on representative historical project data, and the scenario rules should be reviewed by domain experts. The outputs should be treated as decision-support information rather than guaranteed mitigation effects.


In [ ]:
# =========================
# 11.1 Optional New-Project Example
# =========================
# Guarded so Run All succeeds when new_project.csv has not been supplied.

new_project_path = REPO_ROOT / "data" / "new_project.csv"

if new_project_path.exists():
    my_project_df = pd.read_csv(new_project_path)

    missing_features = [
        col for col in features
        if col not in my_project_df.columns
    ]

    if missing_features:
        raise ValueError(
            "new_project.csv is missing required model features: "
            + ", ".join(missing_features)
        )

    new_project_prediction = predict_project_risk(my_project_df)
    all_new_project_tables = []

    for idx in my_project_df.index:
        row_df = my_project_df.loc[[idx]]

        scenarios = recommend_for_project(
            row_df,
            top_n=3,
            max_combo_size=MAX_COMBO_SIZE
        )

        table = recommendations_to_table(scenarios)

        if not table.empty:
            project_identifier = (
                row_df.iloc[0]["Project_ID"]
                if "Project_ID" in row_df.columns
                else f"NewProject_{idx}"
            )

            table.insert(0, "Project_ID", project_identifier)
            all_new_project_tables.append(table)

    if all_new_project_tables:
        new_project_scenarios_df = pd.concat(
            all_new_project_tables,
            ignore_index=True
        )
    else:
        new_project_scenarios_df = pd.DataFrame()

    display(new_project_prediction)
    display(new_project_scenarios_df)

    new_project_scenarios_df.to_csv(
        RESULTS_DIR / "new_project_scenario_results.csv",
        index=False
    )
else:
    print(
        "new_project.csv was not found. "
        "Skipping the optional new-project example."
    )
